# Threshold and Hungarian Matching Viewer

This Voilà-compatible interface provides interactive cross-database atomic energy-level matching without requiring users to edit notebook code.

1. Select an isoelectronic sequence and ion.
2. Select two available databases. Only databases with a `lev_<database>` dataset in the selected HDF5 file are listed.
3. Set the threshold minimum, maximum, and step.
4. Click **Load Viewer**.

Large ions, dense threshold grids, complete change analysis, and complete trace analysis may require additional computation. Large downloads are prepared only when their corresponding button is clicked.


In [ ]:
import gc
import html
import math
import os
import sys
from pathlib import Path

import h5py
import numpy as np
import ipywidgets as widgets
from IPython.display import clear_output, display

# Locate the project root when Voilà starts from either the repository root or viewer/.
cwd = Path.cwd().resolve()
if (cwd / "viewer" / "threshold_viewer.py").exists():
    PROJECT_ROOT = cwd
elif (cwd / "threshold_viewer.py").exists() and cwd.name == "viewer":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = None
    for candidate in [cwd] + list(cwd.parents):
        if (candidate / "viewer" / "threshold_viewer.py").exists():
            PROJECT_ROOT = candidate
            break
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            f"Cannot find the project root from cwd={cwd}. "
            "Expected viewer/threshold_viewer.py under the project root."
        )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dataclean.seq_list import ION_TYPE_PREFIX
from viewer import threshold_viewer as tv

CLEAN_DATA_DIR = PROJECT_ROOT / "clean_data"
THR_VIEW_ROOT = PROJECT_ROOT / "map_data" / "thr_view"
CFG_MODEL_ROOT = PROJECT_ROOT / "map_data" / "cfg_ts"

# Fixed publication settings.
EDGE_ABLATION = "xcos"
XCOS_KEEP = "cfg,phys,energy"
MODEL_PATH = None
STRUCT_BETA = 0
STRUCT_CANDK = 100
STRUCT_TOPK = 1000
STRUCT_TOL = 0.005
STRUCT_WAV_MODE = "log"


In [ ]:
DEFAULT_SEQ = "he-like"
DEFAULT_ION = "020908"
DEFAULT_DB1 = "atomdb313"
DEFAULT_DB2 = "spex30801"


def _prefer(options, preferred, fallback_index=0):
    options = list(options)
    if not options:
        return None
    if preferred in options:
        return preferred
    return options[min(int(fallback_index), len(options) - 1)]


def discover_sequences():
    return [
        sequence
        for sequence in ION_TYPE_PREFIX
        if (CFG_MODEL_ROOT / sequence / "cfg_ts_model.pth").exists()
    ]


def discover_ions(sequence):
    prefix = ION_TYPE_PREFIX[str(sequence)]
    return sorted(path.stem for path in CLEAN_DATA_DIR.glob(f"{prefix}*.h5"))


def discover_databases(ion):
    h5_path = CLEAN_DATA_DIR / f"{ion}.h5"
    if not h5_path.exists():
        return []

    databases = []
    with h5py.File(h5_path, "r") as h5_file:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                leaf = name.split("/")[-1]
                if leaf.startswith("lev_") and not leaf.endswith("_delete"):
                    databases.append(leaf.replace("lev_", "", 1))

        h5_file.visititems(visit)
    return sorted(set(databases))


def build_thresholds(minimum, maximum, step, maximum_count=101):
    values = [float(minimum), float(maximum), float(step)]
    if not all(math.isfinite(value) for value in values):
        raise ValueError("Threshold minimum, maximum, and step must be finite numbers.")
    minimum, maximum, step = values
    if step <= 0:
        raise ValueError("Threshold step must be greater than zero.")
    if minimum > maximum:
        raise ValueError("Threshold minimum cannot exceed threshold maximum.")
    if minimum < 0 or maximum > 1:
        raise ValueError("Threshold values must be within the similarity range [0, 1].")

    tolerance = max(1e-12, abs(step) * 1e-9)
    count = int(math.floor((maximum - minimum) / step + tolerance)) + 1
    thresholds = minimum + step * np.arange(count, dtype=float)
    thresholds = np.round(thresholds, 12)
    thresholds = thresholds[thresholds <= maximum + tolerance]
    if len(thresholds) == 0:
        thresholds = np.array([minimum], dtype=float)
    if abs(float(thresholds[-1]) - maximum) <= tolerance:
        thresholds[-1] = maximum
    if len(thresholds) > int(maximum_count):
        raise ValueError(
            f"The threshold grid contains {len(thresholds)} values; "
            f"the maximum is {maximum_count}. Increase the step or narrow the range."
        )
    return thresholds


sequence_options = discover_sequences()
if not sequence_options:
    raise FileNotFoundError("No sequence-specific configuration models were found.")

# Layout dimensions for the parameter panel.
# Change PANEL_WIDTH to resize the outer panel.
# The controls automatically divide each row into equal-width columns.
PANEL_WIDTH = "640px"
CONTROL_GAP = "10px"

pair_control_layout = widgets.Layout(
    width="auto",
    min_width="0",
    flex="1 1 0",
)
threshold_control_layout = widgets.Layout(
    width="auto",
    min_width="0",
    flex="1 1 0",
)
threshold_control_style = {"description_width": "115px"}
row_layout = widgets.Layout(
    width="100%",
    display="flex",
    flex_flow="row nowrap",
    gap=CONTROL_GAP,
)

seq_w = widgets.Dropdown(
    options=sequence_options,
    value=_prefer(sequence_options, DEFAULT_SEQ),
    description="Sequence",
    layout=pair_control_layout,
)
ion_w = widgets.Dropdown(description="Ion", layout=pair_control_layout)
db1_w = widgets.Dropdown(description="DB1", layout=pair_control_layout)
db2_w = widgets.Dropdown(description="DB2", layout=pair_control_layout)
thr_min_w = widgets.FloatText(
    value=0.00,
    description="Threshold min",
    style=threshold_control_style,
    layout=threshold_control_layout,
)
thr_max_w = widgets.FloatText(
    value=0.99,
    description="Threshold max",
    style=threshold_control_style,
    layout=threshold_control_layout,
)
thr_step_w = widgets.FloatText(
    value=0.01,
    description="Threshold step",
    style=threshold_control_style,
    layout=threshold_control_layout,
)
load_btn = widgets.Button(description="Load Viewer", button_style="success", layout=widgets.Layout(width="150px"))
status_w = widgets.HTML()
viewer_output = widgets.Output()
viewer_reference = {"data": None}


def _refresh_databases(change=None):
    old_db1 = db1_w.value
    old_db2 = db2_w.value
    options = discover_databases(ion_w.value) if ion_w.value else []
    db1_w.options = options
    db2_w.options = options
    db1_w.value = old_db1 if old_db1 in options else _prefer(options, DEFAULT_DB1)
    db2_w.value = old_db2 if old_db2 in options else _prefer(options, DEFAULT_DB2, fallback_index=1)


def _refresh_ions(change=None):
    old_ion = ion_w.value
    options = discover_ions(seq_w.value)
    ion_w.options = options
    preferred = DEFAULT_ION if seq_w.value == DEFAULT_SEQ else None
    ion_w.value = old_ion if old_ion in options else _prefer(options, preferred)
    _refresh_databases()


seq_w.observe(_refresh_ions, names="value")
ion_w.observe(_refresh_databases, names="value")
_refresh_ions()


def _on_load_clicked(_):
    if load_btn.disabled:
        return
    load_btn.disabled = True
    status_w.value = "<span>Loading data and computing similarity...</span>"
    try:
        if not ion_w.value:
            raise ValueError("No ion is available for the selected sequence.")
        if not db1_w.value or not db2_w.value:
            raise ValueError("Two databases are required.")
        if db1_w.value == db2_w.value:
            raise ValueError("DB1 and DB2 must be different.")

        thresholds = build_thresholds(
            thr_min_w.value,
            thr_max_w.value,
            thr_step_w.value,
        )
        cfg_model_path = CFG_MODEL_ROOT / seq_w.value / "cfg_ts_model.pth"
        if not cfg_model_path.exists():
            raise FileNotFoundError(f"Cannot find the configuration model for {seq_w.value}.")

        viewer_reference["data"] = None
        gc.collect()
        with viewer_output:
            clear_output(wait=True)
            viewer_reference["data"] = tv.quick_start_viewer(
                ion=ion_w.value,
                db1=db1_w.value,
                db2=db2_w.value,
                thresholds=thresholds,
                model_path=MODEL_PATH,
                clean_data_dir=CLEAN_DATA_DIR,
                thr_view_root=THR_VIEW_ROOT,
                seq=seq_w.value,
                cfg_model_path=cfg_model_path,
                edge_ablation=EDGE_ABLATION,
                xcos_keep=XCOS_KEEP,
                struct_beta=STRUCT_BETA,
                struct_candk=STRUCT_CANDK,
                struct_topk=STRUCT_TOPK,
                struct_tol=STRUCT_TOL,
                struct_wav_mode=STRUCT_WAV_MODE,
                summarize=True,
                launch=True,
                default_threshold=0.90,
            )
        status_w.value = "<span style='color:#2e6b2e;'>Viewer loaded successfully.</span>"
    except Exception as exc:
        status_w.value = f"<span style='color:#b00020;'>{html.escape(str(exc))}</span>"
    finally:
        load_btn.disabled = False


load_btn.on_click(_on_load_clicked)

controls = widgets.VBox(
    [
        widgets.HBox([seq_w, ion_w], layout=row_layout),
        widgets.HBox([db1_w, db2_w], layout=row_layout),
        widgets.HBox([thr_min_w, thr_max_w, thr_step_w], layout=row_layout),
        widgets.HBox([load_btn, status_w], layout=row_layout),
    ],
    layout=widgets.Layout(
        border="1px solid #cfcfcf",
        padding="10px",
        width=PANEL_WIDTH,
        max_width="100%",
        height="auto",
        box_sizing="border-box",
    ),
)
display(controls, viewer_output)
